In [1]:

# Import libraries
import pandas as pd
import numpy as np

# Load dataset (no header in file)
file_path = "drug_consumption.data"
df = pd.read_csv(file_path, header=None)

print("Shape:", df.shape)
df.head()


Shape: (1885, 32)


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,1,0.49788,0.48246,-0.05921,0.96082,0.12600,0.31287,-0.57545,-0.58331,-0.91699,...,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL2,CL0,CL0
1,2,-0.07854,-0.48246,1.98437,0.96082,-0.31685,-0.67825,1.93886,1.43533,0.76096,...,CL4,CL0,CL2,CL0,CL2,CL3,CL0,CL4,CL0,CL0
2,3,0.49788,-0.48246,-0.05921,0.96082,-0.31685,-0.46725,0.80523,-0.84732,-1.62090,...,CL0,CL0,CL0,CL0,CL0,CL0,CL1,CL0,CL0,CL0
3,4,-0.95197,0.48246,1.16365,0.96082,-0.31685,-0.14882,-0.80615,-0.01928,0.59042,...,CL0,CL0,CL2,CL0,CL0,CL0,CL0,CL2,CL0,CL0
4,5,0.49788,0.48246,1.98437,0.96082,-0.31685,0.73545,-1.63340,-0.45174,-0.30172,...,CL1,CL0,CL0,CL1,CL0,CL0,CL2,CL2,CL0,CL0


In [2]:

# Add proper column names (UCI Drug Consumption dataset standard)
col_names = [
    "ID","Age","Gender","Education","Country","Ethnicity",
    "Nscore","Escore","Oscore","Ascore","Cscore","Impulsive","SS",
    "Alcohol","Amphet","Amyl","Benzos","Caff","Cannabis","Choc","Coke","Crack",
    "Ecstasy","Heroin","Ketamine","Legalh","LSD","Meth","Mushrooms","Nicotine","Semer","VSA"
]
df.columns = col_names

print("Columns assigned:", len(df.columns))
df.head()


Columns assigned: 32


,ID,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,...,Ecstasy,Heroin,Ketamine,Legalh,LSD,Meth,Mushrooms,Nicotine,Semer,VSA
0,1,0.49788,0.48246,-0.05921,0.96082,0.12600,0.31287,-0.57545,-0.58331,-0.91699,...,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL2,CL0,CL0
1,2,-0.07854,-0.48246,1.98437,0.96082,-0.31685,-0.67825,1.93886,1.43533,0.76096,...,CL4,CL0,CL2,CL0,CL2,CL3,CL0,CL4,CL0,CL0
2,3,0.49788,-0.48246,-0.05921,0.96082,-0.31685,-0.46725,0.80523,-0.84732,-1.62090,...,CL0,CL0,CL0,CL0,CL0,CL0,CL1,CL0,CL0,CL0
3,4,-0.95197,0.48246,1.16365,0.96082,-0.31685,-0.14882,-0.80615,-0.01928,0.59042,...,CL0,CL0,CL2,CL0,CL0,CL0,CL0,CL2,CL0,CL0
4,5,0.49788,0.48246,1.98437,0.96082,-0.31685,0.73545,-1.63340,-0.45174,-0.30172,...,CL1,CL0,CL0,CL1,CL0,CL0,CL2,CL2,CL0,CL0


In [3]:

# Basic cleaning checks (missing + duplicates)
print("Missing values per column (should be 0):")
print(df.isnull().sum().sum())

dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)

# If duplicates exist, remove them (safe step)
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after duplicate removal:", df.shape)


Missing values per column (should be 0):
0
Duplicate rows: 0
Shape after duplicate removal: (1885, 32)


In [4]:

# Convert drug usage labels (CL0..CL6) to ordered numeric classes 0..6
drug_cols = col_names[13:]  # all drug columns

class_map = {f"CL{i}": i for i in range(7)}

for c in drug_cols:
    df[c] = df[c].map(class_map).astype(int)

print("Drug columns converted to numeric classes (0..6).")
df[drug_cols].head()


Drug columns converted to numeric classes (0..6).


,Alcohol,Amphet,Amyl,Benzos,Caff,Cannabis,Choc,Coke,Crack,Ecstasy,Heroin,Ketamine,Legalh,LSD,Meth,Mushrooms,Nicotine,Semer,VSA
0,5,2,0,2,6,0,5,0,0,0,0,0,0,0,0,0,2,0,0
1,5,2,2,0,6,4,6,3,0,4,0,2,0,2,3,0,4,0,0
2,6,0,0,0,6,3,4,0,0,0,0,0,0,0,0,1,0,0,0
3,4,0,0,3,5,2,4,2,0,0,0,2,0,0,0,0,2,0,0
4,4,1,1,0,6,3,6,0,0,1,0,0,1,0,0,2,2,0,0


In [5]:
# Cell 6
# Create binary drug-use flags (simple version)
# Here: 0 = Never used (CL0), 1 = Used at least once (CL1..CL6)
for c in drug_cols:
    df[c + "_binary"] = (df[c] >= 1).astype(int)

print("Binary drug-use columns created.")
df[[drug_cols[0], drug_cols[0] + "_binary"]].head()


Binary drug-use columns created.


,Alcohol,Alcohol_binary
0,5,1
1,5,1
2,6,1
3,4,1
4,4,1


In [6]:
# Cell 7
# Final cleaning step: remove duplicates and null values (if any)

print("Before cleaning:")
print("Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

# Remove null values (extra safety, dataset already clean)
df = df.dropna()

# Reset index
df = df.reset_index(drop=True)

print("\nAfter cleaning:")
print("Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())


Before cleaning:
Shape: (1885, 51)
Total Missing Values: 0
Duplicate Rows: 0

After cleaning:
Shape: (1885, 51)
Total Missing Values: 0
Duplicate Rows: 0
